# Mortality-anchored analysis of COVID-19 surveillance

## Reproducible analysis for the *Epidemics* manuscript

This notebook constructs a mortality-anchored incidence benchmark from all-cause excess mortality and compares its wave timing and amplitude with reported COVID-19 cases. The benchmark uses quadratic third-difference regularization, a fixed infection-to-death shift, and a prespecified phase-specific IFR schedule.

The primary timing analysis is restricted to countries with weekly mortality observations. Reported cases and mortality-anchored incidence are smoothed on the same centered seven-day scale. Epidemic peaks are matched within a prespecified 60-day window, and both signed and absolute peak lag are reported. Phase summaries are calculated at the country-phase level, with percentile bootstrap intervals based on countries.

The reconstructed series is conditional on the mortality baseline, regularization parameter, fixed shift, IFR schedule, and treatment of reporting revisions. It is not interpreted as directly observed or uniquely identified infection incidence.

Historical 1918 scenarios and files prepared for previous journal formats are outside the scope of this notebook and are not generated.

Running all cells creates `V22_RESULTS_TO_RETURN.zip`. In Google Colab the archive downloads automatically; in other Jupyter environments a download link is displayed.


In [ ]:
import gc
import hashlib
import json
import os
import platform
import shutil
import sys
import warnings
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
from scipy.interpolate import CubicSpline
from scipy.signal import find_peaks
from scipy.sparse import diags, eye
from scipy.sparse.linalg import spsolve

warnings.filterwarnings("ignore")

OUTPUT_DIR = Path("Epidemics_V22_Results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}; pandas: {pd.__version__}; SciPy: {scipy.__version__}")
print(f"Output directory: {OUTPUT_DIR.resolve()}")


In [ ]:
# =============================================================================
# 1. DATA SOURCES
# =============================================================================
URL_WMD = "https://raw.githubusercontent.com/akarlinsky/world_mortality/main/world_mortality.csv"
URL_JHU_CASES = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_confirmed_global.csv"
URL_JHU_DEATHS = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/csse_covid_19_time_series/time_series_covid19_deaths_global.csv"
URL_JHU_POP = "https://raw.githubusercontent.com/CSSEGISandData/COVID-19/master/csse_covid_19_data/UID_ISO_FIPS_LookUp_Table.csv"

RUN_STARTED_UTC = datetime.now(timezone.utc).isoformat()
print("Downloading source datasets...")
df_wmd_full = pd.read_csv(URL_WMD)
df_jhu_cases_full = pd.read_csv(URL_JHU_CASES)
df_jhu_deaths_full = pd.read_csv(URL_JHU_DEATHS)
df_pop_full = pd.read_csv(URL_JHU_POP)
print("Downloads complete.")

# =============================================================================
# 2. PRIMARY ANALYSIS PARAMETERS
# =============================================================================
PRIMARY_TAU_SHIFT = 20
PRIMARY_LAMBDA_H3 = 5000.0
PRIMARY_IFR_WILD = 0.0105
PRIMARY_IFR_VACCINES = 0.0060
PRIMARY_IFR_OMICRON = 0.0010

TIMING_SMOOTH_DAYS = 7
MATCH_MAX_LAG_DAYS = 60
BOOTSTRAP_REPLICATES = 5000
BOOTSTRAP_SEED = 20260814

START_DATE = pd.Timestamp("2020-02-01")
END_DATE = pd.Timestamp("2022-12-31")
ERA_1_END = pd.Timestamp("2021-01-01")
ERA_2_END = pd.Timestamp("2021-12-01")

NAME_MAPPING = {
    "United States": "US",
    "South Korea": "Korea, South",
    "Taiwan": "Taiwan*",
    "Russia": "Russia",
}

S_c_off_inf, S_c_lat_inf, S_c_fill_inf = "#85C1E9", "#154360", "#D4E6F1"
S_c_off_dead, S_c_lat_dead, S_c_fill_dead = "#F1948A", "#C0392B", "#F5B7B1"


In [ ]:
# =============================================================================
# 3. CORE PROCESSING FUNCTION
# =============================================================================
class CountryExclusion(RuntimeError):
    pass


def _national_population(jhu_country):
    rows = df_pop_full[df_pop_full["Country_Region"] == jhu_country].copy()
    if rows.empty:
        raise CountryExclusion("No JHU population row")

    admin0 = rows.copy()
    if "Province_State" in admin0.columns:
        admin0 = admin0[admin0["Province_State"].isna()]
    if "Admin2" in admin0.columns:
        admin0 = admin0[admin0["Admin2"].isna()]

    values = admin0["Population"].dropna()
    if values.empty or float(values.max()) <= 0:
        raise CountryExclusion("No valid national/Admin0 population")
    return float(values.max())


def _daily_jhu_series(frame, jhu_country):
    rows = frame[frame["Country/Region"] == jhu_country]
    if rows.empty:
        raise CountryExclusion("No matching JHU surveillance series")
    cumulative = rows.iloc[:, 4:].sum(axis=0)
    cumulative.index = pd.to_datetime(cumulative.index, format="%m/%d/%y")
    # Negative daily revisions are truncated at zero under the
    # prespecified surveillance-data preprocessing rule.
    return cumulative.diff().fillna(0).clip(lower=0)


def process_country(
    wmd_country,
    *,
    tau_shift=None,
    lambda_h3=None,
    ifr_wild=None,
    ifr_vaccines=None,
    ifr_omicron=None,
):
    tau_shift = PRIMARY_TAU_SHIFT if tau_shift is None else int(tau_shift)
    lambda_h3 = PRIMARY_LAMBDA_H3 if lambda_h3 is None else float(lambda_h3)
    ifr_wild = PRIMARY_IFR_WILD if ifr_wild is None else float(ifr_wild)
    ifr_vaccines = PRIMARY_IFR_VACCINES if ifr_vaccines is None else float(ifr_vaccines)
    ifr_omicron = PRIMARY_IFR_OMICRON if ifr_omicron is None else float(ifr_omicron)

    jhu_country = NAME_MAPPING.get(wmd_country, wmd_country)
    pop = _national_population(jhu_country)
    daily_cases_official = _daily_jhu_series(df_jhu_cases_full, jhu_country)
    daily_deaths_official = _daily_jhu_series(df_jhu_deaths_full, jhu_country)

    df_wmd_c = df_wmd_full[df_wmd_full["country_name"] == wmd_country].copy()
    if df_wmd_c.empty:
        raise CountryExclusion("No WMD observations")

    units = df_wmd_c["time_unit"].dropna().astype(str).str.lower()
    if units.empty:
        raise CountryExclusion("Missing WMD time unit")
    time_unit = units.mode().iloc[0]
    if time_unit not in {"weekly", "monthly"}:
        raise CountryExclusion(f"Unsupported WMD time unit: {time_unit}")
    df_wmd_c = df_wmd_c[df_wmd_c["time_unit"].astype(str).str.lower() == time_unit]
    timing_eligible = time_unit == "weekly"

    df_base = df_wmd_c[df_wmd_c["year"] < 2020].copy()
    baseline_years = int(df_base["year"].nunique())
    if baseline_years < 2:
        raise CountryExclusion("Fewer than two pre-2020 baseline years")
    baseline = df_base.groupby("time")["deaths"].mean().to_dict()

    df_pand = df_wmd_c[df_wmd_c["year"] >= 2020].copy()
    df_pand["baseline"] = df_pand["time"].map(baseline)
    df_pand["excess"] = df_pand["deaths"] - df_pand["baseline"]
    df_pand = df_pand.dropna(subset=["excess"])
    if len(df_pand) < 12:
        raise CountryExclusion("Insufficient pandemic mortality observations")

    if time_unit == "weekly":
        period_start = pd.to_datetime(
            df_pand["year"].astype(str) + df_pand["time"].astype(str) + "1",
            format="%G%V%u",
        )
        df_pand["Period_Days"] = 7.0
        df_pand["Date"] = period_start + pd.Timedelta(days=3)
    else:
        period_start = pd.to_datetime(
            df_pand["year"].astype(str)
            + "-"
            + df_pand["time"].astype(str)
            + "-01"
        )
        df_pand["Period_Days"] = period_start.dt.days_in_month.astype(float)
        df_pand["Date"] = period_start + pd.to_timedelta(
            (df_pand["Period_Days"] - 1) / 2,
            unit="D",
        )

    df_pand["Daily_Excess_Rate"] = df_pand["excess"] / df_pand["Period_Days"]
    df_pand = (
        df_pand.sort_values("Date")
        .dropna(subset=["Date", "Daily_Excess_Rate"])
        .drop_duplicates(subset=["Date"], keep="last")
    )
    if len(df_pand) < 4:
        raise CountryExclusion("Too few dated mortality observations")

    origin = df_pand["Date"].min()
    numeric_days = (df_pand["Date"] - origin).dt.total_seconds() / 86400.0
    spline = CubicSpline(numeric_days, df_pand["Daily_Excess_Rate"])
    daily_dates = pd.date_range(df_pand["Date"].min(), df_pand["Date"].max(), freq="D")
    eval_days = (daily_dates - origin).days
    raw_daily_deaths = spline(eval_days)

    n = len(raw_daily_deaths)
    if n <= 3:
        raise CountryExclusion("Daily grid too short for third-difference penalty")
    d3 = diags(
        [[-1] * (n - 3), [3] * (n - 2), [-3] * (n - 1), [1] * n],
        [0, 1, 2, 3],
        shape=(n - 3, n),
    )
    system = eye(n) + lambda_h3 * (d3.T @ d3)
    h3_deaths = spsolve(system, raw_daily_deaths)

    analysis_start = max(START_DATE, daily_dates.min())
    analysis_end = min(END_DATE, daily_dates.max())
    if analysis_start >= analysis_end:
        raise CountryExclusion("No overlap with analysis window")

    df_master = pd.DataFrame(
        {
            "Excess_Deaths_Raw_Daily": raw_daily_deaths,
            "Regularized_Excess_Deaths": h3_deaths,
        },
        index=daily_dates,
    ).loc[analysis_start:analysis_end].copy()

    df_master["Official_Deaths"] = daily_deaths_official.reindex(df_master.index).fillna(0)
    df_master["Official_Cases"] = daily_cases_official.reindex(df_master.index).fillna(0)
    df_master["Infection_Date"] = df_master.index - pd.Timedelta(days=tau_shift)

    def get_ifr(date):
        if date < ERA_1_END:
            return ifr_wild
        if date < ERA_2_END:
            return ifr_vaccines
        return ifr_omicron

    df_infections = df_master[["Infection_Date", "Regularized_Excess_Deaths"]].copy()
    df_infections["IFR"] = df_infections["Infection_Date"].map(get_ifr)
    df_infections["Mortality_Anchored_Incidence"] = (
        df_infections["Regularized_Excess_Deaths"] / df_infections["IFR"]
    ).clip(lower=0)
    shifted = df_infections.set_index("Infection_Date")["Mortality_Anchored_Incidence"]
    df_master["Mortality_Anchored_Incidence"] = shifted.reindex(df_master.index).fillna(0)

    factor_100k = pop / 100000.0
    factor_1m = pop / 1000000.0
    df_master["CI14_Official"] = (
        df_master["Official_Cases"].rolling(14, min_periods=1).sum() / factor_100k
    )
    df_master["CI14_Mortality_Anchored"] = (
        df_master["Mortality_Anchored_Incidence"].rolling(14, min_periods=1).sum() / factor_100k
    )
    df_master["Official_Cases_7d"] = (
        df_master["Official_Cases"]
        .rolling(TIMING_SMOOTH_DAYS, center=True, min_periods=1)
        .mean()
    )
    df_master["Mortality_Anchored_Incidence_7d"] = (
        df_master["Mortality_Anchored_Incidence"]
        .rolling(TIMING_SMOOTH_DAYS, center=True, min_periods=1)
        .mean()
    )
    df_master["Official_Deaths_1M"] = df_master["Official_Deaths"] / factor_1m
    df_master["Regularized_Excess_Deaths_1M"] = df_master["Regularized_Excess_Deaths"] / factor_1m

    anchor_peaks_raw, _ = find_peaks(df_master["CI14_Mortality_Anchored"], distance=30)
    valleys, _ = find_peaks(-df_master["CI14_Mortality_Anchored"])
    boundaries = np.sort(np.unique(np.concatenate(([0], valleys, [len(df_master) - 1]))))

    valid_peaks = []
    for peak in anchor_peaks_raw:
        left = boundaries[boundaries < peak].max()
        right = boundaries[boundaries > peak].min()
        segment = df_master.iloc[left : right + 1]
        if segment["CI14_Mortality_Anchored"].max() >= 1000 or segment["Regularized_Excess_Deaths_1M"].max() >= 1.0:
            valid_peaks.append(int(peak))

    official_peak_indices, _ = find_peaks(df_master["Official_Cases_7d"], distance=30)
    official_peak_indices = [
        int(i) for i in official_peak_indices if df_master["Official_Cases_7d"].iloc[i] > 0
    ]
    used_official_peaks = set()
    wave_stats = []

    for wave_number, peak_idx in enumerate(valid_peaks, start=1):
        start_idx = int(boundaries[boundaries < peak_idx].max())
        end_idx = int(boundaries[boundaries > peak_idx].min())
        wave_data = df_master.iloc[start_idx : end_idx + 1]

        w_anchor_cases = float(wave_data["Mortality_Anchored_Incidence"].sum())
        w_official_cases = float(wave_data["Official_Cases"].sum())
        w_anchor_deaths = float(wave_data["Regularized_Excess_Deaths"].sum())
        w_official_deaths = float(wave_data["Official_Deaths"].sum())

        anchor_peak_date = wave_data["Mortality_Anchored_Incidence_7d"].idxmax()
        official_peak_date = pd.NaT
        matched = False
        if timing_eligible and wave_data["Mortality_Anchored_Incidence_7d"].max() > 0:
            candidates = []
            for candidate_idx in official_peak_indices:
                if candidate_idx in used_official_peaks:
                    continue
                candidate_date = df_master.index[candidate_idx]
                distance_days = abs((candidate_date - anchor_peak_date).days)
                if distance_days <= MATCH_MAX_LAG_DAYS:
                    candidates.append((distance_days, candidate_idx, candidate_date))
            if candidates:
                _, selected_idx, official_peak_date = min(candidates, key=lambda item: item[0])
                used_official_peaks.add(selected_idx)
                matched = True

        if matched:
            signed_lag = int((anchor_peak_date - official_peak_date).days)
            absolute_lag = abs(signed_lag)
        else:
            signed_lag = np.nan
            absolute_lag = np.nan

        mortality_peak_h3 = wave_data["Regularized_Excess_Deaths"].idxmax()
        mortality_peak_official = wave_data["Official_Deaths"].idxmax()
        mortality_peak_lag = int((mortality_peak_h3 - mortality_peak_official).days)

        wave_stats.append(
            {
                "Country": wmd_country,
                "Wave_Number": wave_number,
                "Wave_Start": wave_data.index.min().strftime("%Y-%m-%d"),
                "Wave_End": wave_data.index.max().strftime("%Y-%m-%d"),
                "Mortality_Time_Unit": time_unit,
                "Timing_Eligible": timing_eligible,
                "Peak_Matched": matched,
                "Peak_Date_Mortality_Anchored_7d": anchor_peak_date.strftime("%Y-%m-%d"),
                "Peak_Date_Official_7d": (
                    official_peak_date.strftime("%Y-%m-%d") if pd.notna(official_peak_date) else None
                ),
                "Peak_Lag_Days_Cases_Signed": signed_lag,
                "Peak_Lag_Days_Cases_Absolute": absolute_lag,
                "Mortality_Anchored_Cases_Abs": int(round(w_anchor_cases)),
                "Official_Cases_Abs": int(round(w_official_cases)),
                "Regularized_Excess_Deaths_Abs": int(round(w_anchor_deaths)),
                "Official_Deaths_Abs": int(round(w_official_deaths)),
                "Peak_Lag_Days_Deaths": mortality_peak_lag,
                "Tau_Shift_Days": tau_shift,
                "Lambda_H3": lambda_h3,
            }
        )

    total_anchor_cases = float(df_master["Mortality_Anchored_Incidence"].sum())
    total_official_cases = float(df_master["Official_Cases"].sum())
    total_anchor_deaths = float(df_master["Regularized_Excess_Deaths"].sum())
    total_official_deaths = float(df_master["Official_Deaths"].sum())

    case_gap = (
        100 * (1 - total_official_cases / total_anchor_cases)
        if total_anchor_cases > 0
        else np.nan
    )
    death_gap = (
        100 * (1 - total_official_deaths / total_anchor_deaths)
        if total_anchor_deaths > 0
        else np.nan
    )

    country_stat = {
        "Country": wmd_country,
        "Population": int(round(pop)),
        "Baseline_Years": baseline_years,
        "Mortality_Time_Unit": time_unit,
        "Timing_Eligible": timing_eligible,
        "Coverage_Start": df_master.index.min().strftime("%Y-%m-%d"),
        "Coverage_End": df_master.index.max().strftime("%Y-%m-%d"),
        "Total_Waves_Detected": len(valid_peaks),
        "Total_Mortality_Anchored_Cases": int(round(total_anchor_cases)),
        "Total_Official_Cases": int(round(total_official_cases)),
        "Model_Implied_Case_Gap_Pct": round(case_gap, 3) if pd.notna(case_gap) else np.nan,
        "Total_Regularized_Excess_Deaths": int(round(total_anchor_deaths)),
        "Total_Official_Deaths": int(round(total_official_deaths)),
        "Official_Excess_Mortality_Gap_Pct": round(death_gap, 3) if pd.notna(death_gap) else np.nan,
        "Tau_Shift_Days": tau_shift,
        "Lambda_H3": lambda_h3,
    }

    return country_stat, wave_stats, df_master, valid_peaks, boundaries, pop


In [ ]:
# =============================================================================
# 4. POPULATION VALIDATION
# =============================================================================
expected_ranges = {
    "Spain": (40_000_000, 60_000_000),
    "United Kingdom": (55_000_000, 80_000_000),
    "United States": (250_000_000, 400_000_000),
    "Germany": (70_000_000, 100_000_000),
}

for country, (lower, upper) in expected_ranges.items():
    jhu_country = NAME_MAPPING.get(country, country)
    value = _national_population(jhu_country)
    assert lower <= value <= upper, f"Population validation failed for {country}: {value}"
    print(country, int(value))


## Primary analysis

The batch analysis records all country inclusion decisions, generates wave- and country-level metrics, calculates country-phase summaries and bootstrap intervals, and writes the complete result archive.


In [ ]:
# =============================================================================
# 6. BATCH ANALYSIS AND REPRODUCIBLE OUTPUTS
# =============================================================================
wmd_countries = sorted(df_wmd_full["country_name"].dropna().unique())
country_results = []
wave_results = []
inclusion_rows = []

print(f"Processing {len(wmd_countries)} country series from WMD...")
for country in wmd_countries:
    try:
        result = process_country(
            country,
            tau_shift=PRIMARY_TAU_SHIFT,
            lambda_h3=PRIMARY_LAMBDA_H3,
        )
        country_results.append(result[0])
        wave_results.extend(result[1])
        inclusion_rows.append(
            {
                "Country": country,
                "Status": "included",
                "Reason": "",
                "Mortality_Time_Unit": result[0]["Mortality_Time_Unit"],
            }
        )
    except CountryExclusion as exc:
        inclusion_rows.append(
            {
                "Country": country,
                "Status": "excluded",
                "Reason": str(exc),
                "Mortality_Time_Unit": "",
            }
        )
    except Exception as exc:
        inclusion_rows.append(
            {
                "Country": country,
                "Status": "error",
                "Reason": repr(exc),
                "Mortality_Time_Unit": "",
            }
        )
    gc.collect()

df_country = pd.DataFrame(country_results)
df_wave = pd.DataFrame(wave_results)
df_inclusion = pd.DataFrame(inclusion_rows)

assert not df_country.empty, "No countries were processed"
assert not df_wave.empty, "No waves were detected"
assert df_country["Tau_Shift_Days"].eq(PRIMARY_TAU_SHIFT).all()
assert df_wave["Tau_Shift_Days"].eq(PRIMARY_TAU_SHIFT).all()
assert df_country["Lambda_H3"].eq(PRIMARY_LAMBDA_H3).all()
assert df_wave["Lambda_H3"].eq(PRIMARY_LAMBDA_H3).all()

valid_lags = df_wave["Peak_Lag_Days_Cases_Signed"].notna()
assert np.allclose(
    df_wave.loc[valid_lags, "Peak_Lag_Days_Cases_Absolute"],
    df_wave.loc[valid_lags, "Peak_Lag_Days_Cases_Signed"].abs(),
)
assert (
    df_wave.loc[valid_lags, "Mortality_Time_Unit"].eq("weekly").all()
), "Primary timing dataset contains a non-weekly mortality series"

df_wave["Peak_Date_Mortality_Anchored_7d"] = pd.to_datetime(df_wave["Peak_Date_Mortality_Anchored_7d"])

def assign_phase(date):
    if date < ERA_1_END:
        return "2020"
    if date < ERA_2_END:
        return "2021_pre_omicron"
    return "omicron_era"


PHASE_ORDER = ["2020", "2021_pre_omicron", "omicron_era"]
df_wave["Phase"] = df_wave["Peak_Date_Mortality_Anchored_7d"].map(assign_phase)
df_wave["Case_Gap_Pct"] = np.where(
    df_wave["Mortality_Anchored_Cases_Abs"] > 0,
    100 * (1 - df_wave["Official_Cases_Abs"] / df_wave["Mortality_Anchored_Cases_Abs"]),
    np.nan,
)

df_primary = df_wave.loc[
    df_wave["Timing_Eligible"]
    & df_wave["Peak_Matched"]
    & df_wave["Peak_Lag_Days_Cases_Signed"].notna()
].copy()
df_primary["Phase"] = pd.Categorical(
    df_primary["Phase"], categories=PHASE_ORDER, ordered=True
)

country_phase = (
    df_primary.groupby(["Country", "Phase"], observed=True)
    .agg(
        Wave_Count=("Wave_Number", "size"),
        Anchor_Cases=("Mortality_Anchored_Cases_Abs", "sum"),
        Official_Cases=("Official_Cases_Abs", "sum"),
        Median_Signed_Lag=("Peak_Lag_Days_Cases_Signed", "median"),
        Median_Absolute_Lag=("Peak_Lag_Days_Cases_Absolute", "median"),
    )
    .reset_index()
)
country_phase["Case_Gap_Pct"] = np.where(
    country_phase["Anchor_Cases"] > 0,
    100 * (1 - country_phase["Official_Cases"] / country_phase["Anchor_Cases"]),
    np.nan,
)

def q25(series):
    return series.quantile(0.25)


def q75(series):
    return series.quantile(0.75)


phase_summary = (
    country_phase.groupby("Phase", observed=True)
    .agg(
        Countries=("Country", "nunique"),
        Waves=("Wave_Count", "sum"),
        Median_Case_Gap_Pct=("Case_Gap_Pct", "median"),
        Gap_Q25=("Case_Gap_Pct", q25),
        Gap_Q75=("Case_Gap_Pct", q75),
        Median_Signed_Lag=("Median_Signed_Lag", "median"),
        Signed_Lag_Q25=("Median_Signed_Lag", q25),
        Signed_Lag_Q75=("Median_Signed_Lag", q75),
        Median_Absolute_Lag=("Median_Absolute_Lag", "median"),
        Absolute_Lag_Q25=("Median_Absolute_Lag", q25),
        Absolute_Lag_Q75=("Median_Absolute_Lag", q75),
    )
    .reindex(PHASE_ORDER)
    .reset_index()
)

rng = np.random.default_rng(BOOTSTRAP_SEED)

def bootstrap_median(values, replicates=BOOTSTRAP_REPLICATES):
    values = pd.Series(values).dropna().to_numpy(dtype=float)
    if len(values) == 0:
        return np.nan, np.nan, np.nan
    draws = rng.choice(values, size=(replicates, len(values)), replace=True)
    estimates = np.median(draws, axis=1)
    return (
        float(np.median(values)),
        float(np.quantile(estimates, 0.025)),
        float(np.quantile(estimates, 0.975)),
    )


bootstrap_rows = []
for phase in PHASE_ORDER:
    group = country_phase[country_phase["Phase"].astype(str) == phase]
    for metric in ["Median_Signed_Lag", "Median_Absolute_Lag", "Case_Gap_Pct"]:
        estimate, ci_low, ci_high = bootstrap_median(group[metric])
        bootstrap_rows.append(
            {
                "Phase": phase,
                "Metric": metric,
                "Countries": int(group["Country"].nunique()),
                "Estimate": estimate,
                "CI95_Low": ci_low,
                "CI95_High": ci_high,
                "Bootstrap_Replicates": BOOTSTRAP_REPLICATES,
            }
        )
df_bootstrap = pd.DataFrame(bootstrap_rows)

pairwise_rows = []
for phase_a, phase_b in [
    ("2020", "2021_pre_omicron"),
    ("2021_pre_omicron", "omicron_era"),
    ("2020", "omicron_era"),
]:
    for metric in ["Median_Signed_Lag", "Median_Absolute_Lag", "Case_Gap_Pct"]:
        wide = country_phase.pivot(index="Country", columns="Phase", values=metric)
        if phase_a not in wide.columns or phase_b not in wide.columns:
            continue
        paired = wide[[phase_a, phase_b]].dropna()
        differences = paired[phase_b] - paired[phase_a]
        estimate, ci_low, ci_high = bootstrap_median(differences)
        pairwise_rows.append(
            {
                "Phase_A": phase_a,
                "Phase_B": phase_b,
                "Metric": metric,
                "Difference_Definition": "Phase_B minus Phase_A",
                "Paired_Countries": int(len(paired)),
                "Median_Difference": estimate,
                "CI95_Low": ci_low,
                "CI95_High": ci_high,
                "Bootstrap_Replicates": BOOTSTRAP_REPLICATES,
            }
        )
df_pairwise = pd.DataFrame(pairwise_rows)

# -----------------------------------------------------------------------------
# Country-phase timing distributions
# -----------------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True)
positions = np.arange(1, len(PHASE_ORDER) + 1)
labels = ["2020", "2021 pre-Omicron", "Omicron era"]

for ax, metric, ylabel, title in [
    (axes[0], "Median_Signed_Lag", "Signed lag (days)", "Direction of timing divergence"),
    (axes[1], "Median_Absolute_Lag", "Absolute lag (days)", "Magnitude of timing divergence"),
]:
    arrays = [
        country_phase.loc[country_phase["Phase"].astype(str) == phase, metric]
        .dropna()
        .to_numpy()
        for phase in PHASE_ORDER
    ]
    ax.boxplot(arrays, positions=positions, widths=0.55, showfliers=False)
    jitter_rng = np.random.default_rng(BOOTSTRAP_SEED + (0 if metric.endswith("Signed_Lag") else 1))
    for position, values in zip(positions, arrays):
        jitter = jitter_rng.normal(0, 0.045, size=len(values))
        ax.scatter(
            np.full(len(values), position) + jitter,
            values,
            s=13,
            alpha=0.45,
            color="#154360",
        )
    ax.axhline(0, color="grey", linewidth=0.9, linestyle="--")
    ax.set_xticks(positions, labels, rotation=15)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(axis="y", alpha=0.22)

fig.suptitle("Weekly-mortality countries; one summary per country and phase")
fig.tight_layout()
figure_pdf = OUTPUT_DIR / "V22_Figure1_country_phase_timing.pdf"
figure_png = OUTPUT_DIR / "V22_Figure1_country_phase_timing.png"
fig.savefig(figure_pdf, bbox_inches="tight")
fig.savefig(figure_png, dpi=300, bbox_inches="tight")
plt.show()

# -----------------------------------------------------------------------------
# Export analysis tables and figures
# -----------------------------------------------------------------------------
output_frames = {
    "V22_country_metrics.csv": df_country,
    "V22_wave_metrics.csv": df_wave,
    "V22_primary_weekly_timing_waves.csv": df_primary,
    "V22_country_phase_metrics.csv": country_phase,
    "V22_phase_summary_weekly_only.csv": phase_summary,
    "V22_phase_bootstrap_country.csv": df_bootstrap,
    "V22_phase_pairwise_changes.csv": df_pairwise,
    "V22_inclusion_log.csv": df_inclusion,
}
for filename, frame in output_frames.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

phase_codes = {
    "2020": "Early",
    "2021_pre_omicron": "Middle",
    "omicron_era": "Late",
}
macro_lines = [
    "% Generated from the country-phase summary",
    rf"\newcommand{{\Ncountries}}{{{df_country['Country'].nunique()}}}",
    rf"\newcommand{{\Nwaves}}{{{len(df_wave)}}}",
    rf"\newcommand{{\NwavesTiming}}{{{len(df_primary)}}}",
]
for row in phase_summary.dropna(subset=["Phase"]).itertuples(index=False):
    code = phase_codes[str(row.Phase)]
    macro_lines.extend(
        [
            rf"\newcommand{{\Ncountries{code}}}{{{int(row.Countries)}}}",
            rf"\newcommand{{\Nwaves{code}}}{{{int(row.Waves)}}}",
            rf"\newcommand{{\Gap{code}}}{{{row.Median_Case_Gap_Pct:.1f}}}",
            rf"\newcommand{{\Lag{code}}}{{{row.Median_Signed_Lag:.1f}}}",
            rf"\newcommand{{\AbsLag{code}}}{{{row.Median_Absolute_Lag:.1f}}}",
        ]
    )
(OUTPUT_DIR / "V22_phase_macros.tex").write_text("\n".join(macro_lines) + "\n", encoding="utf-8")

weekly_waves = df_wave[df_wave["Timing_Eligible"]]
match_rate = 100 * weekly_waves["Peak_Matched"].mean() if len(weekly_waves) else np.nan
manifest = {
    "run_started_utc": RUN_STARTED_UTC,
    "run_finished_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {
        "numpy": np.__version__,
        "pandas": pd.__version__,
        "scipy": scipy.__version__,
    },
    "data_sources": {
        "world_mortality": URL_WMD,
        "jhu_cases": URL_JHU_CASES,
        "jhu_deaths": URL_JHU_DEATHS,
        "jhu_population": URL_JHU_POP,
    },
    "parameters": {
        "tau_shift_days": PRIMARY_TAU_SHIFT,
        "lambda_h3": PRIMARY_LAMBDA_H3,
        "ifr_wild": PRIMARY_IFR_WILD,
        "ifr_vaccines": PRIMARY_IFR_VACCINES,
        "ifr_omicron": PRIMARY_IFR_OMICRON,
        "timing_smooth_days": TIMING_SMOOTH_DAYS,
        "peak_match_max_lag_days": MATCH_MAX_LAG_DAYS,
        "bootstrap_replicates": BOOTSTRAP_REPLICATES,
        "bootstrap_seed": BOOTSTRAP_SEED,
        "analysis_start": str(START_DATE.date()),
        "analysis_end": str(END_DATE.date()),
    },
    "counts": {
        "wmd_country_labels": len(wmd_countries),
        "included_countries": int(len(df_country)),
        "weekly_timing_countries": int(df_country["Timing_Eligible"].sum()),
        "all_detected_waves": int(len(df_wave)),
        "weekly_detected_waves": int(len(weekly_waves)),
        "matched_primary_waves": int(len(df_primary)),
        "weekly_peak_match_rate_pct": float(match_rate),
        "batch_errors": int((df_inclusion["Status"] == "error").sum()),
    },
    "estimand_note": (
        "The mortality-anchored incidence benchmark is conditional on the "
        "baseline, fixed infection-to-death shift, regularization parameter, "
        "and infection fatality ratio scenario."
    ),
}
(OUTPUT_DIR / "V22_run_manifest.json").write_text(
    json.dumps(manifest, indent=2), encoding="utf-8"
)

outputs_description = """# Analysis outputs

The archive contains the country inclusion log, country-level metrics, wave-level
metrics, the weekly primary timing sample, country-phase summaries, percentile
bootstrap intervals, paired phase contrasts, a reproducibility manifest, LaTeX
macros, and the principal timing figure in PDF and PNG formats.

The mortality-anchored incidence benchmark is conditional on the specified
baseline, quadratic third-difference regularization, fixed 20-day shift, and
phase-specific IFR schedule. The output is not a direct estimate of uniquely
identified infection incidence.
"""
(OUTPUT_DIR / "README_outputs.md").write_text(outputs_description, encoding="utf-8")

archive_members = [
    OUTPUT_DIR / name for name in output_frames
] + [
    figure_pdf,
    figure_png,
    OUTPUT_DIR / "V22_phase_macros.tex",
    OUTPUT_DIR / "V22_run_manifest.json",
    OUTPUT_DIR / "README_outputs.md",
]

checksum_lines = []
for path in sorted(archive_members, key=lambda item: item.name):
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    checksum_lines.append(f"{digest}  {path.name}")
checksum_file = OUTPUT_DIR / "V22_checksums.sha256"
checksum_file.write_text("\n".join(checksum_lines) + "\n", encoding="utf-8")
archive_members.append(checksum_file)

return_zip = Path("V22_RESULTS_TO_RETURN.zip")
with zipfile.ZipFile(return_zip, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(archive_members, key=lambda item: item.name):
        archive.write(path, arcname=path.name)

print("\nAnalysis complete.")
print(phase_summary.to_string(index=False))
print(f"\nArchive: {return_zip.resolve()}")

try:
    from IPython.display import FileLink, display

    display(FileLink(str(return_zip), result_html_prefix="Download results: "))
except ImportError:
    pass

if "google.colab" in sys.modules:
    from google.colab import files

    files.download(str(return_zip))
